# DPO-Guard — Training on Google Colab (T4)

Run each cell in order. First: **Runtime → Change runtime type → GPU (T4)**.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers==4.44.2 trl==0.11.1 peft==0.12.0 accelerate==0.33.0 bitsandbytes==0.43.3 datasets==2.20.0 pyyaml python-dotenv

## 1. Upload the project zip
Upload `dpo-guard.zip` (the whole project folder zipped).

In [ ]:
from google.colab import files
import os, zipfile
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/')
for root, dirs, _ in os.walk('/content'):
    if 'configs' in dirs and 'src' in dirs:
        os.chdir(root); break
print('CWD =', os.getcwd())

## 2. Hugging Face login (required for Mistral, optional for Phi-2)

In [ ]:
from huggingface_hub import login
import getpass
login(getpass.getpass('HF token: '))

## 3. Split seeds into train/test if you did not already

In [ ]:
import os
if not (os.path.exists('data/train.jsonl') and os.path.exists('data/test.jsonl')):
    !python src/split_dataset.py --input data/construction/preference_pairs_v1.jsonl --train data/train.jsonl --test data/test.jsonl

## 4. Sweep training across 5 beta values
Each beta trains ~45–90 min on T4. Adapters saved to `models/adapters/beta_<value>/`.

In [ ]:
for beta in [0.1, 0.3, 0.5, 0.7, 1.0]:
    print(f'\n===== training beta={beta} =====')
    !python src/train_dpo.py --config configs/dpo_config.yaml --beta {beta}

## 5. Download the trained adapters

In [ ]:
import shutil
shutil.make_archive('/content/dpo_adapters', 'zip', 'models/adapters')
from google.colab import files
files.download('/content/dpo_adapters.zip')

## 6. (Optional) Evaluate + Pareto plot inside Colab

In [ ]:
!python src/evaluate.py --adapters-dir models/adapters --base-model microsoft/phi-2 --output results/metrics.json
!python src/plot_pareto.py --metrics results/metrics.json --output results/pareto_frontier.png
from google.colab import files
files.download('results/metrics.json')
files.download('results/pareto_frontier.png')